In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy
from tqdm import tqdm

from skzeros import AAA, Rectangle, force_subdivide, poles_residues
from skzeros._domain import get_leaf_regions

plt.style.use("style.mplstyle")

In [ ]:
rng = np.random.default_rng(1212)
n = 50
x = -0.4 + 0.8 * rng.integers(0, 1, n, endpoint=True)
A = scipy.linalg.circulant(x)

In [ ]:
e = np.linalg.eigvals(A)

In [ ]:
rng = np.random.default_rng(1234)
u, v = rng.random((2, n))
count = 0


@np.vectorize
def f(z):
    global count  # noqa: PLW0603
    count += getattr(z, "size", 1)
    return u.T @ np.linalg.solve(A - z * np.eye(n), v)


@np.vectorize
def f_z(z):
    return u.T @ np.linalg.solve(
        A - z * np.eye(n), scipy.linalg.solve(A - z * np.eye(n), v)
    )

In [ ]:
count = 0
d = Rectangle(complex(-5.1, -4.9), complex(5, 4.7))
force_subdivide(d, 6)
regions = get_leaf_regions(d)
poles = np.array([])
for region in tqdm(regions):
    res = AAA(lambda z: f_z(z) / f(z), region)
    p, r = poles_residues(*res, residue=True)
    bl, tr = region.bottom_left, region.top_right
    to_keep = (
        (p.real >= bl.real)
        & (p.real <= tr.real)
        & (p.imag >= bl.imag)
        & (p.imag <= tr.imag)
        & (r < 0)
        & (abs(r.real - np.round(r.real)) < 1e-3)
    )
    poles = np.concatenate((poles, p[to_keep]))

In [ ]:
fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(4, 2))
d.plot(ax1)
ax1.plot(poles.real, poles.imag, ".")
ax1.set_xlabel(r"$\Re(z)$")
ax1.set_ylabel(r"$\Im(z)$")
ax1.set_aspect("equal", "box")
ax2.hist(np.log10(np.min(np.abs(np.subtract.outer(e, poles)), axis=-1)))
ax2.set_xlim([-16, -5])
ax2.set_xlabel("Log Error")
ax2.set_ylabel("Frequency")
plt.tight_layout()
plt.savefig("figures/example3b.pdf")

In [ ]:
AAA_err = np.min(np.abs(np.subtract.outer(e, poles)), axis=-1)
print(
    f"AAA: {count=}, max error={np.max(AAA_err):.2e}, min error={np.min(AAA_err):.2e}, medium error={np.median(AAA_err):.2e}"
)
print(len(get_leaf_regions(d)))